In [1]:
%cd ..

/home/hsc/Projects/personal/disertation


In [2]:
from pathlib import Path


DATA_DIRPATH = Path("./data")
PROJECT_DIRPATH = Path("./iterations")
TRAIN_DATA_DIRPATH = DATA_DIRPATH / "train"
TEST_DATA_DIRPATH = DATA_DIRPATH / "test"
TRAIN_CSV_FILEPATH = DATA_DIRPATH / "train_labels.csv"
THRESH_CSV_FILEPATH = DATA_DIRPATH / "train_thresholds.csv"

In [3]:
import torch

DEVICE = torch.device("cuda:0")

In [4]:
import dataclasses
import os

import numpy as np
import pandas as pd


@dataclasses.dataclass
class Prediction:
    image_id: (
        str | None
    )  # A unique identifier for the row -- unused otherwise. Used only on the hidden test set.
    dataset: str
    filename: str
    image_filepath: Path
    cluster_index: int | None = None
    rotation: np.ndarray | None = None
    translation: np.ndarray | None = None


# Set is_train=True to run the notebook on the training data.
# Set is_train=False if submitting an entry to the competition (test data is hidden, and different from what you see on the "test" folder).
is_train = True
data_dir = DATA_DIRPATH
workdir = PROJECT_DIRPATH
os.makedirs(workdir, exist_ok=True)

if is_train:
    sample_submission_csv = DATA_DIRPATH / "train_labels.csv"
    df = pd.read_csv(sample_submission_csv)
    df["image_id"] = df.dataset + "_" + df.image
    DATASET_DIR = TRAIN_DATA_DIRPATH
else:
    sample_submission_csv = DATA_DIRPATH / "sample_submission.csv"
    df = pd.read_csv(sample_submission_csv)
    DATASET_DIR = TEST_DATA_DIRPATH

samples = {}
for _, row in df.iterrows():
    # Note: For the test data, the "scene" column has no meaning, and the rotation_matrix and translation_vector columns are random.
    if row.dataset not in samples:
        samples[row.dataset] = []
    samples[row.dataset].append(
        Prediction(
            image_id=row.image_id,
            dataset=row.dataset,
            filename=row.image,
            image_filepath=DATASET_DIR / row.dataset / row.image,
        )
    )

for dataset in samples:
    print(f'Dataset "{dataset}" -> num_images={len(samples[dataset])}')

Dataset "imc2023_haiper" -> num_images=54
Dataset "imc2023_heritage" -> num_images=209
Dataset "imc2023_theather_imc2024_church" -> num_images=76
Dataset "imc2024_dioscuri_baalshamin" -> num_images=138
Dataset "imc2024_lizard_pond" -> num_images=214
Dataset "pt_brandenburg_british_buckingham" -> num_images=225
Dataset "pt_piazzasanmarco_grandplace" -> num_images=168
Dataset "pt_sacrecoeur_trevi_tajmahal" -> num_images=225
Dataset "pt_stpeters_stpauls" -> num_images=200
Dataset "amy_gardens" -> num_images=200
Dataset "fbk_vineyard" -> num_images=163
Dataset "ETs" -> num_images=22
Dataset "stairs" -> num_images=51


In [5]:
from mts.pipeline.pipeline.imc2025 import IMC2025Pipeline

/home/hsc/Projects/personal/disertation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from __future__ import annotations
from mts.core.types import PathLike


class NoIterationException(BaseException):
    """Raised in case there is no iteration"""


class Project:
    def __init__(
        self,
        project_dir: PathLike,
        iteration_name: str,
        create: bool = True,
    ) -> None:
        self.project_dir = project_dir
        self.iteration_name = iteration_name
        if create:
            self._create()

    @property
    def iteration_dirpath(self) -> Path:
        return self.project_dir / self.iteration_name

    def _create(self):
        self.iteration_dirpath.mkdir(exist_ok=True)

    @classmethod
    def _make_first_iter(cls, resolution: int = 4) -> str:
        return cls._make_iteration_name(1, resolution=resolution)

    @staticmethod
    def _make_iteration_name(index: int, resolution: int = 4) -> str:
        return f"{index:0{resolution}d}"

    @staticmethod
    def _get_last_iter(iterations_dirpaths: list[Path]) -> str:
        iterations = []
        for iteration_dirpath in iterations_dirpaths:
            try:
                iteration_num = int(iteration_dirpath.name)
            except ValueError:
                pass
            else:
                iterations.append((iteration_num, iteration_dirpath))

        if len(iterations) == 0:
            raise NoIterationException(
                "There is no automatically created iteration",
            )
        last_iteration_dirpath: Path = max(iterations, key=lambda x: x[0])[1]
        return last_iteration_dirpath.name

    @classmethod
    def from_next_iteration(
        cls,
        project_dirpath: PathLike,
        resolution: int = 4,
        **kwargs,
    ) -> Project:
        project_dirpath = Path(project_dirpath)
        iterations_dirpaths = list(project_dirpath.glob("*"))
        if len(iterations_dirpaths) == 0:
            last_iteration_name = cls._make_first_iter(resolution)
        else:
            try:
                last_iteration_name = cls._get_last_iter(iterations_dirpaths)
                last_iteration = int(last_iteration_name) + 1
                last_iteration_name = cls._make_iteration_name(last_iteration, resolution)
            except NoIterationException:
                last_iteration_name = cls._make_first_iter(resolution)
        return cls(project_dirpath, last_iteration_name, **kwargs)

    @classmethod
    def from_last_iteration(
        cls,
        project_dirpath: PathLike,
        **kwargs,
    ) -> Project:
        project_dirpath = Path(project_dirpath)
        iterations_dirpaths = list(project_dirpath.glob("*"))
        if len(iterations_dirpaths) == 0:
            raise NoIterationException("There is no iteration ")
        last_iteration_name = cls._get_last_iter(iterations_dirpaths)
        return cls(project_dirpath, last_iteration_name, **kwargs)

In [7]:
from hydra.utils import instantiate
from omegaconf import OmegaConf

In [8]:
cfg = OmegaConf.load("config/pipeline/imc2025/0001.yaml")

In [9]:
from mts.pipeline.step.base import BasePipelineStep


def from_hydra_config(config_filepath: PathLike) -> list[BasePipelineStep]:
    cfg = OmegaConf.load(config_filepath)
    pipeline_steps = []
    for step_name, step in cfg.pipeline["steps"].items():
        pipeline_steps.append(instantiate(step))
    return pipeline_steps

In [10]:
from mts.pipeline.repository.inmemeory import ImageRepository

predictions = samples["imc2023_haiper"]
image_repository = ImageRepository()

for pred in predictions:
    image_repository.add_image(pred.image_filepath)

In [11]:
last_project_iteration = Project.from_next_iteration("iterations")

In [12]:
state = {
    "images_dir": ".",
    "colmap_dirpath": last_project_iteration.iteration_dirpath,
}

In [13]:
from mts.core.types import StateType
from mts.pipeline.repository.inmemeory import ImageRepository


def create_repository(dataset_name: str) -> ImageRepository:
    image_repository = ImageRepository()
    image_repository.add_repository_metadata(dataset_name=dataset_name)
    return image_repository


def create_pipeline(
    pipeline_config_filepath: PathLike,
    image_repository: ImageRepository,
    dataset_name: str,
) -> list[BasePipelineStep]:
    pipeline_steps = from_hydra_config(pipeline_config_filepath)
    return pipeline_steps


def create_pipeline_state(
    imc2025_pipeline: IMC2025Pipeline,
    image_repository: ImageRepository,
    dataset_name: str,
) -> StateType:
    dataset_dirpath = Path(imc2025_pipeline.project_dirpath) / dataset_name
    dataset_dirpath.mkdir(exist_ok=True)
    state = {
        "images_dir": ".",
        "colmap_dirpath": dataset_dirpath,
    }
    return state

In [15]:
pipeline_steps = from_hydra_config("config/pipeline/imc2025/0001.yaml")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loaded LightGlue model


In [19]:
samples["imc2023_haiper"]

[Prediction(image_id='imc2023_haiper_fountain_image_116.png', dataset='imc2023_haiper', filename='fountain_image_116.png', image_filepath=PosixPath('data/train/imc2023_haiper/fountain_image_116.png'), cluster_index=None, rotation=None, translation=None),
 Prediction(image_id='imc2023_haiper_fountain_image_108.png', dataset='imc2023_haiper', filename='fountain_image_108.png', image_filepath=PosixPath('data/train/imc2023_haiper/fountain_image_108.png'), cluster_index=None, rotation=None, translation=None),
 Prediction(image_id='imc2023_haiper_fountain_image_101.png', dataset='imc2023_haiper', filename='fountain_image_101.png', image_filepath=PosixPath('data/train/imc2023_haiper/fountain_image_101.png'), cluster_index=None, rotation=None, translation=None),
 Prediction(image_id='imc2023_haiper_fountain_image_082.png', dataset='imc2023_haiper', filename='fountain_image_082.png', image_filepath=PosixPath('data/train/imc2023_haiper/fountain_image_082.png'), cluster_index=None, rotation=None,

In [20]:
from tqdm.auto import tqdm

image_repository = create_repository("imc2023_haiper")
for sample in tqdm(samples["imc2023_haiper"]):
    image_repository.add_image(sample.image_filepath)

100%|██████████| 54/54 [00:00<00:00, 219044.89it/s]


In [16]:
from functools import partial


imc2025_pipeline = IMC2025Pipeline(
    last_project_iteration.iteration_dirpath,
    samples,
    create_repository,
    lambda *args: pipeline_steps[:-1],
    create_pipeline_state=create_pipeline_state,
)

In [21]:
dataset_name = "imc2023_haiper"
dataset_dirpath = Path(last_project_iteration.iteration_dirpath) / dataset_name
dataset_dirpath.mkdir(exist_ok=True)
state = {
    "images_dir": ".",
    "colmap_dirpath": dataset_dirpath,
}

In [22]:
from mts.pipeline.step.base import run_pipeline


run_pipeline(
    steps=pipeline_steps[:-1],
    image_repository=image_repository,
    input=None,
    state=state,
)

Match the keypoints and descriptors: 100%|██████████| 1431/1431 [00:36<00:00, 38.94it/s]


In [ ]:
pipeline_steps[-1].run(
    image_repository=image_repository,
    input=None,
    state=state,
)

In [17]:
imc2025_pipeline.run(datasets_names=["imc2023_haiper"])

Match the keypoints and descriptors: 100%|██████████| 1431/1431 [00:34<00:00, 41.70it/s]


KeyError: PosixPath('data/train/imc2023_haiper/fountain_image_116.png')

In [69]:
last_project_iteration.iteration_dirpath

PosixPath('iterations/0001')

In [62]:
Project.from_next_iteration("iterations")

'0001'

In [65]:
last_project_iteration.

In [10]:
from pathlib import Path

In [11]:
iter_path = Path("iterations")

In [19]:
(iter_path / "0001").name

'0001'

In [11]:
f"{0:04d}"

'0000'

In [ ]:
if len(iterations) == 0:
    pass

In [ ]:
IMC2025Pipeline()

In [7]:
from mts.core.extractor.aliked import AlikedExtractor


aliked = AlikedExtractor(1024, 1024)

In [9]:
import torch


CUDA0 = torch.device("cuda:0")

In [10]:
from mts.core.embedder.dinov2 import DinoV2GlobalDescriptors


dino_v2 = DinoV2GlobalDescriptors.from_pretrained("weights/dinov2/base")

/home/hsc/Projects/personal/disertation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [11]:
from mts.pipeline.step.extract.kp.base import TorchExtractStep


extract_step = TorchExtractStep(image_repository, aliked)
extract_step = extract_step.to(DEVICE)

In [12]:
from mts.pipeline.step.extract.embedding.base import GlobalDescriptorStep

global_extractor_step = GlobalDescriptorStep(image_repository, dino_v2)
global_extractor_step = global_extractor_step.to(DEVICE)

In [13]:
with torch.no_grad():
    extract_step.run()

Extract keypoints and descriptors: 100%|██████████| 54/54 [00:08<00:00,  6.69it/s]


In [14]:
with torch.no_grad():
    global_extractor_step.run()

Extract keypoints and descriptors: 100%|██████████| 54/54 [00:05<00:00, 10.22it/s]


In [ ]:
from mts.pipeline.step.pair.cdist import CrossEmbeddingParerStep


parer = CrossEmbeddingParerStep(image_repository)

In [16]:
parer.run()

In [17]:
from mts.core.matcher.lg import LightGlueMatcher


lg_matcher = LightGlueMatcher.from_config(
    "aliked",
    {
        "width_confidence": -1,
        "depth_confidence": -1,
        "mp": True if "cuda" in str(DEVICE) else False,
    },
)
lg_matcher = lg_matcher.to(CUDA0)

Loaded LightGlue model


In [18]:
from mts.pipeline.step.match.base import MatchingStep


matching_step = MatchingStep(image_repository, lg_matcher)
matching_step = matching_step.to(DEVICE)

In [19]:
with torch.no_grad():
    matching_step.run()

  0%|          | 0/1431 [00:00<?, ?it/s]

100%|██████████| 1431/1431 [00:35<00:00, 40.44it/s]


In [20]:
import more_itertools as mit
import itertools as it

In [ ]:
from mts.pipeline.step.base import BasePipelineStep


class ColmapReconstruction(BasePipelineStep):
    def run(self) -> None:
        return super().run()

In [ ]:
from abc import ABC, abstractmethod
from copy import deepcopy
from typing import Any
import pycolmap
from mts.core.geometry.rigid3d import Rigid3D
from mts.core.types import PathLike, StateType
from mts.helpers.colmap.database import COLMAPDatabase
from mts.pipeline.repository.export.colmap import export_to_colmap
from mts.pipeline.step.base import BasePipelineStep, use_image_repository, use_params





In [23]:
import hloc

In [24]:
from mts.helpers.colmap.database import COLMAPDatabase

In [ ]:
from mts.pipeline.step.reconstruct.colmap.importer import ColmapImportStep

reconstruction_dirpath = Path("iterations/test")
db_filepath = reconstruction_dirpath / "reconstruction.db"

colmap_import_step = ColmapImportStep.from_db_filepath(db_filepath, image_repository)

In [26]:
colmap_import_step.run()

Add matches: 100%|██████████| 1431/1431 [00:00<00:00, 346618.68it/s]


In [27]:
reconstruction_step = ReconstructionStep(db_filepath, reconstruction_dirpath, ".")

In [ ]:
from typing import Any
import logging

from mts.core.types import StateType
from mts.pipeline.step.base import run_pipeline

LOGGER = logging.getLogger(__name__)


class ReconstructionPipeline(BasePipelineStep):
    def __init__(
        self,
        repository: ImageRepository,
        steps: list[BasePipelineStep],
    ) -> None:
        super().__init__()
        self.steps = steps
        self.repository = repository

    def set_repository(self, repository: ImageRepository) -> None:
        LOGGER.info("Setup repository for the whole pipeline")
        for step in self.steps:
            step.set_image_repository(repository)

    def run(
        self,
        *,
        input: Any | None,
        state: StateType | None = None,
    ) -> Any:
        self._set_up()
        result = run_pipeline(self.steps, input=input, state=state)
        return result

In [ ]:
class IMC2025Pipeline:
    def __init__(
        self,
        samples: dict[str, list[Prediction]],
        pipeline_step: BasePipelineStep,
    ) -> None:
        pass

In [ ]:
from mts.helpers.colmap.options import create_incremental_pipeline_options


create_incremental_pipeline_options

In [28]:
result = reconstruction_step.run()

I20251220 21:35:28.270785 140169457874496 misc.cc:44] 
Feature matching & geometric verification
I20251220 21:35:28.272157 140169344710208 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.272658 140169294353984 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.272893 140169210492480 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.273425 140169311139392 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.273650 140169193707072 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.274550 140169449481792 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.274603 140169302746688 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.274624 140169432696384 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.274652 140169202099776 sift.cc:1452] Creating SIFT CPU feature matcher
I20251220 21:35:28.274576 140169327924800 sift.cc:1452] Creating SIFT CPU feature matcher
I20

In [29]:
from hloc.utils import viz as img_viz

In [30]:
import kornia.feature as KF

In [ ]:
st_idx, nd_idx = 42, 54
img_viz.plot_images(
    [image_repository.load_image(st_idx), image_repository.load_image(nd_idx)]
)
st_kp = image_repository.get_keypoints(st_idx)
nd_kp = image_repository.get_keypoints(nd_idx)

img_viz.plot_keypoints([st_kp, nd_kp])
matches = image_repository.get_matches(st_idx, nd_idx)
img_viz.plot_matches(st_kp[matches[:, st_idx]], nd_kp[matches[:, nd_idx]])

KeyError: 54